In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as pl
import matplotlib.animation as animation
from IPython.display import HTML

from scipy.interpolate import make_interp_spline
from scipy.stats import norm
from scipy.special import factorial
from numpy.polynomial.hermite_e import hermeval

import powerbox as pbox
import shoddy

In [ ]:
def use_style(dark=True):
    pl.rcdefaults()
    pl.style.use(['custom.mplstyle', 'dark_theme.mplstyle'] if dark else ['custom.mplstyle'])
    pl.rcParams['animation.embed_limit'] = 80.0   # MB, for to_jshtml()

use_style(dark=False)

In [ ]:
z     = 9.0
model = shoddy.Model(z=z)

mcl, ls = model.limber_cl(model.matter_power_spectrum)
w_nobias, th_in = model.cf_ang(power_func=model.matter_power_spectrum)
# log-log interpolant so powerbox can evaluate C_ell at any multipole k = ell
_logcl = make_interp_spline(np.log(ls), np.log(np.clip(mcl, 1e-30, None)))
lmin, lmax = ls[0], ls[-1]
def Cl(k):
    k = np.clip(np.atleast_1d(k), lmin, lmax)
    return np.exp(_logcl(np.log(k)))

w_spl = make_interp_spline(th_in * 60., w_nobias)

In [ ]:
# Parameters

L_BOX   = np.deg2rad(10)
N_GRID  = 4096      # grid size [pixels]
SEED    = 2027
RSMOOTH = 2         # Gaussian smoothing scale [pixels]

NREAL = 6           # realizations averaged into the measured xi
NBIN  = 16          # radial bins for xi(theta)
JMAX  = 60          # terms kept in the Kaiser series

nus = np.linspace(-0.5, 2.5, 61)    # thresholds swept, in units of sigma

PIX = np.rad2deg(L_BOX / N_GRID) * 60.   # arcmin per pixel
print(f"pixel = {PIX:.3f}', smoothing = {RSMOOTH * PIX:.2f}', box = {np.rad2deg(L_BOX):.1f} deg")

In [ ]:
def bias(nu, sigma=None):
    '''Kaiser threshold bias b(nu) for a Gaussian field of rms sigma.'''
    sigma = SIGMA if sigma is None else sigma
    return norm.pdf(nu) / norm.sf(nu) / sigma


def kaiser_series(nu, xi, sigma=None, jmax=JMAX):
    '''Exact xi of the excursion set g > nu*sigma, from the Gaussian xi.'''
    sigma = SIGMA if sigma is None else sigma
    w = xi / sigma**2
    j = np.arange(1, jmax + 1)
    he = np.array([hermeval(nu, [0] * (n - 1) + [1]) for n in j])   # He_{j-1}(nu)
    coef = (norm.pdf(nu) * he)**2 / (factorial(j) * norm.sf(nu)**2)
    return np.sum(coef[:, None] * w[None, :]**j[:, None], axis=0)


def bias_from_threshold(t, sigma=None):
    '''nu, b, and area fraction for an absolute threshold t on the smoothed Gaussian field.'''
    sigma = SIGMA if sigma is None else sigma
    nu = t / sigma
    return nu, norm.pdf(nu) / norm.sf(nu) / sigma, norm.sf(nu)

In [ ]:
bias_from_threshold(0.01, 0.0672)

In [ ]:
box = pbox.LogNormalPowerBox(boxlength=L_BOX, N=N_GRID, dim=2, seed=SEED, pk=Cl)
field = box.delta_x()
gauss_field = np.log1p(field)

kx = np.fft.fftfreq(N_GRID)
ky = np.fft.rfftfreq(N_GRID)
W = np.exp(-0.5 * (kx[:, None]**2 + kx[None, :]**2) * (RSMOOTH * 2 * np.pi)**2)
smooth_kernel = np.exp(-0.5 * (kx[:, None]**2 + ky[None, :]**2) * (RSMOOTH * 2 * np.pi)**2)
smooth_field = np.fft.irfft2(np.fft.rfft2(gauss_field) * smooth_kernel, s=(N_GRID, N_GRID))
smooth_field -= smooth_field.mean()
print(smooth_field.std())

ix  = np.fft.fftfreq(N_GRID) * N_GRID
sep  = np.sqrt(ix[:, None]**2 + ix[None, :]**2) * PIX
rbin = np.geomspace(RSMOOTH * PIX, N_GRID / 15 * PIX, NBIN + 1)

ibin   = np.digitize(sep, rbin) - 1
valid  = (ibin >= 0) & (ibin < NBIN)
counts = np.bincount(ibin[valid], minlength=NBIN)
rmid   = np.bincount(ibin[valid], weights=sep[valid], minlength=NBIN) / counts

thresh_field = (smooth_field > 0.01).astype(float)
thresh_field = thresh_field/thresh_field.mean() - 1

corr = np.fft.irfft2(np.abs(np.fft.rfft2(thresh_field))**2, s=(N_GRID, N_GRID)) / thresh_field.size
acf = np.bincount(ibin[valid], weights=corr[valid], minlength=NBIN) / counts

In [ ]:
P = np.fft.fft2(np.fft.ifftshift(box.gaussian_correlation_array())) * W**2   # W applied once per copy
P[0, 0] = 0.0
corr_exact = np.real(np.fft.ifft2(P))
acf_exact = np.bincount(ibin[valid], weights=corr_exact[valid], minlength=NBIN) / counts

In [ ]:
pl.imshow(corr[:40, :40], origin='lower', norm='log')
pl.colorbar()
pl.show()

In [ ]:
pl.loglog(rmid / 60, acf)
pl.show()

In [ ]:
pl.imshow(field+1)
pl.colorbar()
pl.show()

In [ ]:
pl.imshow(smooth_field)
pl.colorbar()
pl.show()

In [ ]:
box.gaussian_correlation_array().mean()

In [ ]:
#pl.loglog(np.rad2deg(np.arange(N_GRID//2)*L_BOX/N_GRID), box.gaussian_correlation_array()[N_GRID//2,N_GRID//2:])
#pl.loglog(th_in, w_nobias)
pl.loglog(rmid/60, acf)
pl.loglog(rmid/60, acf_exact * 13.45**2)
#pl.xlim(1e-1, 1)
#pl.ylim(1e-4, 1e-2)
pl.show()

In [ ]:
th_in*3600

## Measuring $\xi$ of the field and of the thresholded field

Same estimator as `galaxy_bias.ipynb`, moved onto the angular lognormal box: FFT the periodic
field, square, transform back, and bin radially — separations now in **arcmin** rather than
pixels, via `PIX = L_BOX / N_GRID`.

Two things change relative to the Gaussian case:

- **Threshold on the Gaussian field, not on $\delta$.** `LogNormalPowerBox` gives a lognormal
  $\delta$; the field behind it is $g = \ln(1+\delta)$. The transform is monotonic, so
  $\delta > \delta_c$ and $g > \nu\sigma_g$ select the *identical* excursion set — but only $g$
  is Gaussian, so only $\xi_g$ feeds the Kaiser series. We measure both and check the closure
  $\xi_\delta = e^{\xi_g} - 1$.
- **$\sigma \neq 1$.** The field keeps its physical amplitude, so
  $b(\nu) = \sigma^{-1}\varphi(\nu)/[1-\Phi(\nu)]$ and the series argument is
  $w = \xi_g/\sigma^2$.

The field is Gaussian-smoothed on `RSMOOTH` pixels first: without it the excursion set is set by
pixel noise, and there is no separation of scales for a linear-bias plateau to live in.
Tracer $\xi$ comes from the 0/1 mask (no shot noise, no exclusion), as before.

In [ ]:
# --- Gaussian smoothing kernel, in cycles-per-pixel Fourier units
_kx = np.fft.fftfreq(N_GRID)[:, None]
_ky = np.fft.rfftfreq(N_GRID)[None, :]
_smooth = np.exp(-0.5 * (_kx**2 + _ky**2) * (RSMOOTH * 2 * np.pi)**2)


def smoothed(f):
    '''Gaussian-smooth a periodic field on scale RSMOOTH and remove its mean.'''
    s = np.fft.irfft2(np.fft.rfft2(f) * _smooth, s=(N_GRID, N_GRID))
    return s - s.mean()


# --- radial binning of the real-space separation grid, in arcmin
_ix  = np.fft.fftfreq(N_GRID) * N_GRID
sep  = np.sqrt(_ix[:, None]**2 + _ix[None, :]**2) * PIX
rbin = np.geomspace(RSMOOTH * PIX, N_GRID / 20 * PIX, NBIN + 1)

ibin   = np.digitize(sep, rbin) - 1
valid  = (ibin >= 0) & (ibin < NBIN)
counts = np.bincount(ibin[valid], minlength=NBIN)
rmid   = np.bincount(ibin[valid], weights=sep[valid], minlength=NBIN) / counts


def xi_of(f):
    '''Correlation function of a zero-mean periodic field, radially binned via FFT.'''
    corr = np.fft.irfft2(np.abs(np.fft.rfft2(f))**2, s=(N_GRID, N_GRID)) / f.size
    return np.bincount(ibin[valid], weights=corr[valid], minlength=NBIN) / counts

In [ ]:
gfields  = []
xi_gauss = np.zeros(NBIN)     # xi of the smoothed Gaussian field g
xi_delta = np.zeros(NBIN)     # xi of the smoothed lognormal overdensity

for i in range(NREAL):
    b = pbox.LogNormalPowerBox(boxlength=L_BOX, N=N_GRID, dim=2, seed=SEED + i, pk=Cl)
    d = b.delta_x()
    g = smoothed(np.log1p(d))     # the Gaussian field behind the lognormal
    gfields.append(g.astype(np.float32))
    xi_gauss += xi_of(g)
    xi_delta += xi_of(smoothed(d))

xi_gauss /= NREAL
xi_delta /= NREAL
SIGMA = float(np.mean([g.std() for g in gfields]))

print(f"{NREAL} realizations of {N_GRID}^2, sigma_g = {SIGMA:.4f}")
print(f"xi_g spans {xi_gauss.min():.3e} .. {xi_gauss.max():.3e}"
      f" over theta = {rmid[0]:.1f}' .. {rmid[-1]:.1f}'")
print(f"lognormal closure, max |xi_d / (e^xi_g - 1) - 1| = "
      f"{np.abs(xi_delta / np.expm1(xi_gauss) - 1).max():.2e}")

In [ ]:
xi_tracer = np.zeros((len(nus), NBIN))

for i, nu in enumerate(nus):
    acc = np.zeros(NBIN)
    for g in gfields:
        m = (g > nu * SIGMA).astype(np.float64)
        acc += xi_of(m / m.mean() - 1.0)
    xi_tracer[i] = acc / NREAL

print("done")

In [ ]:
# sanity check: the plateau of xi_tr / xi_g should approach b^2(nu)
plateau = (rmid > 6 * rmid[0]) & (rmid < 0.8 * rmid[-1])

print(f"plateau over theta = {rmid[plateau][0]:.1f}' .. {rmid[plateau][-1]:.1f}'")
print(" nu     b(nu)    b^2       measured plateau   ratio   f(>nu)")
for i in range(0, len(nus), 10):
    meas = (xi_tracer[i] / xi_gauss)[plateau].mean()
    b2 = bias(nus[i])**2
    print(f"{nus[i]:5.2f}  {bias(nus[i]):7.2f}  {b2:9.1f}  {meas:16.1f}  {meas / b2:7.3f}"
          f"  {norm.sf(nus[i]) * 100:6.2f}%")

### Caveats specific to this box

- *Integral constraint.* $w(\theta)$ at $z=9$ is nearly flat across 10 deg, so most of the
  correlation lives on scales larger than the box. Subtracting the box mean removes it, and the
  measured $\xi_g$ falls well below `w_spl(rmid)` and decays faster. The **ratio**
  $\xi_{\rm tr}/\xi_g$ is unaffected (both estimators lose the same constant), which is why the
  bias test above works even though the absolute amplitude does not match theory.
- *Dynamic range.* The linear-bias plateau needs $\xi_g \ll \sigma^2$, i.e. separations well
  above `RSMOOTH` and well below the box. That window is `N_GRID / RSMOOTH` wide — at
  1024 / 4 it is comfortable; at the original 512 / 10 there is essentially no plateau.
- *Rare tracers.* At $\nu = 2.5$ only ~0.6% of the area is above threshold and the outer bins
  are noisy even averaged over `NREAL` realizations.